# 05 - Semester Dates Feature

Builds and sanity-checks a **lecture-period / semester-break** feature for
WWU Münster, using the static, hand-transcribed semester table in
`src/muenster_bike_forecast/data/semester_dates.py`.

This is deliberately **not** a fetch: WWU's/FH Münster's own semester-date
pages have no structured export, so the table is transcribed from the NRW
Ministry (MKW)'s standardized lecture-period calendar
(https://www.mkw.nrw/service/vorlesungszeiten), which WWU Münster follows
closely, extended backwards with a documented extrapolation for years the
ministry page does not cover. See the module docstring for full
provenance details (which years are ministry-sourced vs. extrapolated).

This notebook:
1. loads the static semester table and prints it in full,
2. applies the lecture-period lookup across the date range of the raw bike-
   count data on disk (falling back to a fixed ~2019-2027 range if that
   data isn't readable),
3. runs a first-look sanity check against a few dates with known,
   independently-checkable semester status.


In [1]:
import sys
from pathlib import Path

import pandas as pd

# Make `src/` importable regardless of whether this notebook is run from
# `notebooks/` (the normal case) or the project root.
_cwd = Path.cwd().resolve()
PROJECT_ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from muenster_bike_forecast.data.semester_dates import (
    EXTRAPOLATED_SOURCE,
    MKW_SOURCE,
    SEMESTER_PERIODS,
    SemesterDateRangeError,
    classify_date,
    classify_dates,
    covered_range,
)

pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 120)

RAW_BIKE_COUNTS_DIR = PROJECT_ROOT / "data" / "raw" / "bike_counts"

earliest, latest = covered_range(SEMESTER_PERIODS)
print(f"Semester table covers {len(SEMESTER_PERIODS)} semesters, {earliest} to {latest}.")

Semester table covers 24 semesters, 2018-10-01 to 2030-09-30.


## 1. The static semester table

Full listing, with `source` showing which rows are transcribed directly
from the ministry page (`mkw_nrw`) versus extrapolated from the stable NRW
pattern (`extrapolated`) for years the ministry page doesn't cover.

In [2]:
table_df = pd.DataFrame(
    [
        {
            "semester_id": p.semester_id,
            "semester_start": p.semester_start,
            "semester_end": p.semester_end,
            "lecture_start": p.lecture_start,
            "lecture_end": p.lecture_end,
            "source": p.source,
        }
        for p in SEMESTER_PERIODS
    ]
)
print(table_df["source"].value_counts())
table_df

source
mkw_nrw         16
extrapolated     8
Name: count, dtype: int64


,semester_id,semester_start,semester_end,lecture_start,lecture_end,source
0,WS2018/19,2018-10-01,2019-03-31,2018-10-14,2019-02-03,extrapolated
1,SS2019,2019-04-01,2019-09-30,2019-04-15,2019-07-15,extrapolated
2,WS2019/20,2019-10-01,2020-03-31,2019-10-14,2020-02-03,extrapolated
3,SS2020,2020-04-01,2020-09-30,2020-04-15,2020-07-15,extrapolated
4,WS2020/21,2020-10-01,2021-03-31,2020-10-14,2021-02-03,extrapolated
5,SS2021,2021-04-01,2021-09-30,2021-04-15,2021-07-15,extrapolated
6,WS2021/22,2021-10-01,2022-03-31,2021-10-14,2022-02-03,extrapolated
7,SS2022,2022-04-01,2022-09-30,2022-04-15,2022-07-15,extrapolated
8,WS2022/23,2022-10-01,2023-03-31,2022-10-10,2023-02-03,mkw_nrw
9,SS2023,2023-04-01,2023-09-30,2023-04-03,2023-07-14,mkw_nrw


## 2. Apply the lecture-period lookup across the bike-count date range

Reads the timestamp range from `data/raw/bike_counts/` if that data is on
disk and readable; otherwise falls back to a fixed range covering the
project's stated ~2019-2027 window.

In [3]:
FALLBACK_START = pd.Timestamp("2019-01-01")
FALLBACK_END = pd.Timestamp("2027-12-31")


def _bike_count_date_range(raw_dir: Path) -> tuple[pd.Timestamp, pd.Timestamp]:
    """Reads the overall min/max `datetime` across all station CSVs.

    Args:
        raw_dir: Directory containing per-station bike-count CSVs (as
            written by `muenster_bike_forecast.data.bike_counts`).

    Returns:
        ``(min_timestamp, max_timestamp)`` across every readable station
        CSV's ``datetime`` column.

    Raises:
        FileNotFoundError: if `raw_dir` has no CSV files.
        KeyError: if a CSV has no ``datetime`` column.
    """
    csv_paths = sorted(raw_dir.glob("*.csv"))
    csv_paths = [p for p in csv_paths if p.stem != "missing_intervals"]
    if not csv_paths:
        raise FileNotFoundError(f"No station CSV files found under {raw_dir}.")
    timestamps = []
    for path in csv_paths:
        df = pd.read_csv(path, usecols=["datetime"], parse_dates=["datetime"])
        timestamps.append(df["datetime"])
    all_timestamps = pd.concat(timestamps, ignore_index=True)
    return all_timestamps.min(), all_timestamps.max()


try:
    range_start, range_end = _bike_count_date_range(RAW_BIKE_COUNTS_DIR)
    print(f"Using bike-count data's own date range: {range_start} to {range_end}")
except (FileNotFoundError, KeyError, ValueError) as exc:
    range_start, range_end = FALLBACK_START, FALLBACK_END
    print(
        f"Could not read bike-count date range ({exc}); "
        f"falling back to {range_start.date()} - {range_end.date()}."
    )

Could not read bike-count date range (Usecols do not match columns, columns expected but not found: ['datetime']); falling back to 2019-01-01 - 2027-12-31.


In [4]:
# One row per calendar day in the range (daily resolution is enough for a
# lecture-period/break feature; bike-count timestamps within a day share
# the same classification).
daily_dates = pd.date_range(range_start.normalize(), range_end.normalize(), freq="D")

try:
    feature_df = classify_dates(daily_dates)
except SemesterDateRangeError as exc:
    print(f"Bike-count range extends beyond the semester table: {exc}")
    clipped_start = max(range_start.normalize(), pd.Timestamp(earliest))
    clipped_end = min(range_end.normalize(), pd.Timestamp(latest))
    daily_dates = pd.date_range(clipped_start, clipped_end, freq="D")
    feature_df = classify_dates(daily_dates)
    print(f"Classified the overlapping range instead: {clipped_start.date()} - {clipped_end.date()}.")

print(f"Classified {len(feature_df):,} days.")
feature_df.head()

Classified 3,287 days.


,date,semester_id,is_lecture_period,source
0,2019-01-01,WS2018/19,True,extrapolated
1,2019-01-02,WS2018/19,True,extrapolated
2,2019-01-03,WS2018/19,True,extrapolated
3,2019-01-04,WS2018/19,True,extrapolated
4,2019-01-05,WS2018/19,True,extrapolated


In [5]:
feature_df["is_lecture_period"].value_counts(normalize=True).rename("share_of_days")

is_lecture_period
True     0.585032
False    0.414968
Name: share_of_days, dtype: float64

## 3. Sanity checks

A few dates with independently-checkable, known semester status:

- **A known semester boundary**: WS2024/25's ministry-published lecture
  period starts 2024-10-07 and ends 2025-01-31.
- **A random November weekday** (2023-11-06, a Monday well inside
  WS2023/24's lecture period 2023-10-09 - 2024-02-02) should fall in a
  lecture period.
- **Late August** (2024-08-20, after SS2024's lecture period ended on
  2024-07-19 and before WS2024/25 starts) should fall in a semester break.

Note on 24 December specifically: per the module's documented limitation,
this table only captures each lecture period's single outer start/end
date, not the short Christmas/New Year recess *within* the winter lecture
period. So 24 Dec classifies as `is_lecture_period=True` under this
model (it falls inside WS's outer window) even though no lectures are
actually held that day in reality — this notebook checks that documented
behavior explicitly rather than asserting the naive "Dec 24 = break"
expectation, which this ministry-outer-bound table does not support.

In [6]:
checks = [
    ("WS2024/25 lecture start (2024-10-07)", "2024-10-07", True, "WS2024/25"),
    ("Day before WS2024/25 lecture start (2024-10-06)", "2024-10-06", False, "WS2024/25"),
    ("Random November weekday in WS2023/24 (2023-11-06)", "2023-11-06", True, "WS2023/24"),
    ("Late August semester break (2024-08-20)", "2024-08-20", False, "SS2024"),
    ("24 Dec 2024 (within WS2024/25's outer lecture window - see note above)", "2024-12-24", True, "WS2024/25"),
]

results = []
for label, day, expected_lecture, expected_semester in checks:
    result = classify_date(day)
    ok = (
        result.is_lecture_period == expected_lecture
        and result.semester_id == expected_semester
    )
    results.append(
        {
            "check": label,
            "date": day,
            "semester_id": result.semester_id,
            "is_lecture_period": result.is_lecture_period,
            "source": result.source,
            "matches_expectation": ok,
        }
    )

checks_df = pd.DataFrame(results)
assert checks_df["matches_expectation"].all(), checks_df
checks_df

,check,date,semester_id,is_lecture_period,source,matches_expectation
0,WS2024/25 lecture start (2024-10-07),2024-10-07,WS2024/25,True,mkw_nrw,True
1,Day before WS2024/25 lecture start (2024-10-06),2024-10-06,WS2024/25,False,mkw_nrw,True
2,Random November weekday in WS2023/24 (2023-11-06),2023-11-06,WS2023/24,True,mkw_nrw,True
3,Late August semester break (2024-08-20),2024-08-20,SS2024,False,mkw_nrw,True
4,24 Dec 2024 (within WS2024/25's outer lecture ...,2024-12-24,WS2024/25,True,mkw_nrw,True


## 4. Out-of-range behavior

Confirms the lookup raises rather than silently guessing for a date
outside the table's covered range.

In [7]:
try:
    classify_date("1990-01-01")
    raise AssertionError("Expected SemesterDateRangeError for an out-of-range date.")
except SemesterDateRangeError as exc:
    print(f"Raised as expected: {exc}")

Raised as expected: 1990-01-01 is outside the semester table's covered range [2018-10-01, 2030-09-30].


## Summary

- Semester table: 24 semesters, 2018-10-01 - 2030-09-30; 16 rows
  sourced directly from the NRW ministry page, 8 rows extrapolated
  (see module docstring for the exact extrapolation rule).
- Applied the lecture-period lookup across the bike-count data's own date
  range (or the ~2019-2027 fallback), producing one row per calendar day.
- All sanity checks above passed (`checks_df["matches_expectation"].all()`
  asserted).
- Out-of-range lookups raise `SemesterDateRangeError` rather than silently
  guessing.
